# Predicción de Tasa de Interés — Crédito de Consumo Bancario

## Objetivo
Predecir la **tasa de interés efectiva promedio del siguiente mes** para créditos de **Consumo**
otorgados por **Establecimientos Bancarios** en Colombia.

**Fuente:** Superintendencia Financiera de Colombia — API Datos Abiertos Gov.co

---

## Flujo del notebook
| Bloque | Descripción |
|--------|-------------|
| 1 | Importar librerías |
| 2 | Descarga de datos (API) |
| 3 | Inspección inicial del dataset |
| 4 | Limpieza y validación |
| 5 | Filtrado por variables objetivo |
| 6 | Construcción de la serie temporal |
| 7 | EDA — Estadísticas descriptivas |
| 8 | EDA — Serie temporal y tendencia |
| 9 | EDA — Distribución y Boxplot |
| 10 | EDA — Variación mensual |
| 11 | EDA — Comparación entre bancos |
| 12 | EDA — Autocorrelación (ACF) |
| 13 | EDA — Prueba de estacionariedad |
| 14 | EDA — Medias móviles |
| 15 | División Train / Test |
| 16 | Modelo 1: Regresión Lineal con Lags |
| 17 | Modelo 2: ARIMA |
| 18 | Comparación de modelos |
| 19 | Predicción del siguiente mes |
| 20 | Conclusiones |


## Bloque 1 — Importar Librerías

In [1]:
import pandas as pd
import numpy as np
import requests
import time

from datetime import date
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size']      = 11

print(" Librerías cargadas correctamente")


 Librerías cargadas correctamente


## Bloque 2 — Descarga de Datos desde la API

La API solo devuelve **1000 filas por consulta**. Hacemos varias consultas con `$offset`
para descargar el histórico completo. Aplicamos los filtros directamente en la API
con `$where` para no traer datos innecesarios.

- **Filtro 1:** `nombre_tipo_entidad = 'BC-ESTABLECIMIENTO BANCARIO'`
- **Filtro 2:** `tipo_de_cr_dito = 'Consumo'`
- **Filtro 3:** `producto_de_cr_dito = 'Libre inversión'`
- **Filtro 4:** `fecha_corte = 'A elección'`
- **Filtro 5:** `Bancos = 'Bancos pertenecientes al nicho de microcreditos'`


In [5]:
# ── Librerías necesarias ─────────────────────────────────────────────────
import requests
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta
import time

# ── Parámetros de consulta ───────────────────────────────────────────────
URL_API      = "https://www.datos.gov.co/resource/w9zh-vetq.json"
TIPO_ENTIDAD = "BC-ESTABLECIMIENTO BANCARIO"
TIPO_CREDITO = "Consumo"
PRODUCTO     = "Libre inversión"
FECHA_INICIO = date(2023, 10, 1)
FECHA_FIN    = date(2026, 2, 28)

# ── Bancos a excluir ─────────────────────────────────────────────────────
BANCOS_EXCLUIR = (
    "'Bancamia S.A.',"
    "'Banco Contactar',"
    "'Banco Mundo Mujer S.A.',"
    "'Banco Santander',"
    "'Banco W S.A.',"
    "'Mibanco S.A.',"
    "'Coopcentral',"
    "'Banco Pichincha S.A.',"
    "'Bancien',"
    "'Banco Davibank',"
    "'Scotiabank Colpatria S.A.',"
    "'Banco GNB Sudameris',"
    "'Banagrario'"
)

print("=" * 65)
print("  DESCARGANDO DATOS AGREGADOS MES A MES")
print("=" * 65)

dfs = []
fecha_actual = FECHA_INICIO
session    = requests.Session()   # ← sesión reutilizable

while fecha_actual <= FECHA_FIN:

    mes_inicio = fecha_actual.strftime("%Y-%m-%dT00:00:00.000")
    mes_fin    = (fecha_actual + relativedelta(months=1) - relativedelta(days=1)).strftime("%Y-%m-%dT00:00:00.000")
    

    filtro = (
        f"nombre_tipo_entidad = '{TIPO_ENTIDAD}' AND "
        f"tipo_de_cr_dito = '{TIPO_CREDITO}' AND "
        f"producto_de_cr_dito = '{PRODUCTO}' AND "
        f"nombre_entidad NOT IN ({BANCOS_EXCLUIR}) AND "
        f"rango_monto_desembolsado IS NOT NULL AND "
        f"fecha_corte >= '{mes_inicio}' AND "
        f"fecha_corte <= '{mes_fin}' AND "
        f"tasa_efectiva_promedio >= '10'"
    )

    try:
        r = session.get(URL_API, params={ # ← session.get en lugar de requests.get
            "$select": "nombre_entidad, "
                       "date_trunc_ym(fecha_corte) AS mes, "
                       "rango_monto_desembolsado, "                           # ← nuevo
                       "sum(tasa_efectiva_promedio * numero_de_creditos) AS suma_tasa_credito, "
                       "sum(numero_de_creditos) AS total_creditos",
            "$where" : filtro,
            "$group" : "nombre_entidad, date_trunc_ym(fecha_corte), rango_monto_desembolsado",  # ← nuevo
            "$order" : "nombre_entidad, date_trunc_ym(fecha_corte), rango_monto_desembolsado",  # ← nuevo
            "$limit" : 50000
        }, timeout=180)

        data = r.json()

        if len(data) > 0:
            dfs.append(pd.DataFrame(data))
            print(f"  {fecha_actual.strftime('%Y-%m')}: {len(data):>3} combinaciones banco-rango")
        else:
            print(f"  {fecha_actual.strftime('%Y-%m')}: sin datos")

    except Exception as e:
        print(f"  {fecha_actual.strftime('%Y-%m')}: ERROR → {e}")

    fecha_actual += relativedelta(months=1)
    time.sleep(0.3)

df_ponderada = pd.concat(dfs, ignore_index=True)
df_ponderada['suma_tasa_credito'] = pd.to_numeric(df_ponderada['suma_tasa_credito'])
df_ponderada['total_creditos']    = pd.to_numeric(df_ponderada['total_creditos'])
df_ponderada['tasa_ponderada']    = df_ponderada['suma_tasa_credito'] / df_ponderada['total_creditos']
df_ponderada['mes']               = pd.to_datetime(df_ponderada['mes']).dt.to_period('M')

print("-" * 65)
print(f"  Bancos únicos          : {df_ponderada['nombre_entidad'].nunique()}")
print(f"  Meses únicos           : {df_ponderada['mes'].nunique()}")
print(f"  Rangos únicos          : {df_ponderada['rango_monto_desembolsado'].nunique()}")
print(f"  Total combinaciones    : {len(df_ponderada)}")
print("=" * 65)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df_ponderada.round(2)

  DESCARGANDO DATOS AGREGADOS MES A MES
  2023-10: 125 combinaciones banco-rango
  2023-11: 122 combinaciones banco-rango
  2023-12: 127 combinaciones banco-rango
  2024-01: 118 combinaciones banco-rango
  2024-02: 123 combinaciones banco-rango
  2024-03: 123 combinaciones banco-rango
  2024-04: 119 combinaciones banco-rango
  2024-05: 122 combinaciones banco-rango
  2024-06: 125 combinaciones banco-rango
  2024-07: 122 combinaciones banco-rango
  2024-08: 124 combinaciones banco-rango
  2024-09: 122 combinaciones banco-rango
  2024-10: 122 combinaciones banco-rango
  2024-11: 122 combinaciones banco-rango
  2024-12: 122 combinaciones banco-rango
  2025-01: 121 combinaciones banco-rango
  2025-02: 127 combinaciones banco-rango
  2025-03: 127 combinaciones banco-rango
  2025-04: 124 combinaciones banco-rango
  2025-05: 130 combinaciones banco-rango
  2025-06: 124 combinaciones banco-rango
  2025-07: 126 combinaciones banco-rango
  2025-08: 123 combinaciones banco-rango
  2025-09: 126 co

,nombre_entidad,mes,rango_monto_desembolsado,suma_tasa_credito,total_creditos,tasa_ponderada
0,AV Villas,2023-10,Hasta 1 SMLMV,160.46,7,22.92
1,AV Villas,2023-10,Mayor a 100 SMLMV menor o igual a 150 SMLMV,498.80,26,19.18
2,AV Villas,2023-10,Mayor a 12 SMLMV menor o igual a 25 SMLMV,9488.08,369,25.71
3,AV Villas,2023-10,Mayor a 150 SMLMV menor o igual a 300 SMLMV,118.34,5,23.67
4,AV Villas,2023-10,Mayor a 1 SMLMV menor o igual a 3 SMLMV,5958.64,168,35.47
5,AV Villas,2023-10,Mayor a 25 SMLMV menor o igual a 50 SMLMV,6422.63,273,23.53
6,AV Villas,2023-10,Mayor a 3 SMLMV menor o igual a 6 SMLMV,7501.92,234,32.06
7,AV Villas,2023-10,Mayor a 50 SMLMV menor o igual a 100 SMLMV,2997.86,142,21.11
8,AV Villas,2023-10,Mayor a 6 SMLMV menor o igual a 12 SMLMV,9092.28,314,28.96
9,Banco Caja Social S.A.,2023-10,Hasta 1 SMLMV,30100.59,770,39.09


In [7]:
# ── Identificar bancos con meses faltantes ───────────────────────────────

todos_los_meses = df_ponderada['mes'].unique()
todos_los_bancos = df_ponderada['nombre_entidad'].unique()

print("MESES POR BANCO")
print("=" * 60)
for banco in sorted(todos_los_bancos):
    meses_banco = df_ponderada[df_ponderada['nombre_entidad'] == banco]['mes'].nunique()
    estado = " 29 meses" if meses_banco == 29 else f"OJO:solo contiene registros de {meses_banco} meses"
    print(f"  {banco:<30} {estado}")
print("=" * 60)

MESES POR BANCO
  AV Villas                       29 meses
  BBVA Colombia                   29 meses
  Banco Caja Social S.A.          29 meses
  Banco Davivienda                29 meses
  Banco Falabella S.A.            29 meses
  Banco Popular                   29 meses
  Banco Serfinanza S.A.           29 meses
  Banco Unión                     29 meses
  Banco de Bogotá                 29 meses
  Banco de Occidente              29 meses
  Bancolombia                     29 meses
  Bancoomeva                      29 meses
  Finandina                       29 meses
  Itaú                            29 meses
  Lulo Bank                       29 meses


In [9]:
# ── Filtrar bancos con al menos 6 meses de datos ─────────────────────────

bancos_validos = (df_ponderada
    .groupby('nombre_entidad')['mes']
    .nunique()
    .reset_index()
    .rename(columns={'mes': 'n_meses'})
    .query('n_meses >= 29')
    ['nombre_entidad']
    .tolist()
)

df_ponderada = df_ponderada[df_ponderada['nombre_entidad'].isin(bancos_validos)].copy()

print("BANCOS CONSERVADOS (solo con registros en los 29 meses)")
print("=" * 60)
for banco in sorted(bancos_validos):
    n = df_ponderada[df_ponderada['nombre_entidad'] == banco]['mes'].nunique()
    print(f"  {banco:<30} {n} meses")
print("-" * 60)
print(f"  Total bancos conservados : {len(bancos_validos)}")
print(f"  Total de tasas           : {len(df_ponderada)}")
print("=" * 60)

BANCOS CONSERVADOS (solo con registros en los 29 meses)
  AV Villas                      29 meses
  BBVA Colombia                  29 meses
  Banco Caja Social S.A.         29 meses
  Banco Davivienda               29 meses
  Banco Falabella S.A.           29 meses
  Banco Popular                  29 meses
  Banco Serfinanza S.A.          29 meses
  Banco Unión                    29 meses
  Banco de Bogotá                29 meses
  Banco de Occidente             29 meses
  Bancolombia                    29 meses
  Bancoomeva                     29 meses
  Finandina                      29 meses
  Itaú                           29 meses
  Lulo Bank                      29 meses
------------------------------------------------------------
  Total bancos conservados : 15
  Total de tasas           : 3583


In [11]:
df_ponderada.round(2).to_csv("tasas_ponderadas_por_rango_2023-2026.csv", index=False, encoding="utf-8-sig")
print(" Archivo CSV guardado: tasas_ponderadas_por_rango_2023-2026.csv")

 Archivo CSV guardado: tasas_ponderadas_por_rango_2023-2026.csv
